In [ ]:
! pip install torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [ ]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
import xgboost, lightgbm, catboost

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import config as cnf 
import utils as utl

In [ ]:
df_train = pd.read_csv(cnf.path['train_path'])

# Обработка данных
df_train_priz = df_train[['Survived', 'Pclass', 'Age', 'Sex',  'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')

for col in ['Sex', 'Embarked']:
    df_train_priz[col] = df_train_priz[col].astype('category')

X = df_train_priz.drop('Survived', axis=1)
# Второй датасет с OneHotCoding (Не для бустингов)
X = pd.get_dummies(X, dtype=float, drop_first=True)

results = []
SK_Fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)

array_train_accuracy = []
array_test_accuracy = []

model = 